# Data Exploration

Dataset: CMU Book Summary Dataset — http://www.cs.cmu.edu/~dbamman/booksummaries.html

Plot summaries of 16,559 books extracted from English Wikipedia (Nov. 2012), with Freebase metadata.

Each step below uses the **same functions as the application pipeline** (`src/data/data.py`), so the corpus studied here is exactly the one used by the app.

In [1]:
# Make the project root importable (notebooks live in notebooks/)
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

## Imports

In [2]:
import numpy as np
import pandas as pd

# Functions shared with the application pipeline
from src.data.data import (
    load_config, transform_format_data, parse_genre,
    download_require_nltk_resources, preprocess_text, run,
)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

config = load_config(ROOT / "configs/data.yaml")

## Load the raw `txt` file

Tab-separated file without header. `quoting=3` (QUOTE_NONE) because summaries contain quotes.

In [3]:
df = transform_format_data(ROOT / config["paths"]["raw_txt_data"], config["dataset"]["columns"])
print(f"Loaded {len(df)} books")
df.info()

Loaded 16559 books
<class 'pandas.DataFrame'>
RangeIndex: 16559 entries, 0 to 16558
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   wikipedia_id      16559 non-null  int64
 1   freebase_id       16559 non-null  str  
 2   title             16559 non-null  str  
 3   author            14177 non-null  str  
 4   publication_date  10949 non-null  str  
 5   genres            12841 non-null  str  
 6   synopsis          16559 non-null  str  
dtypes: int64(1), str(6)
memory usage: 42.1 MB


In [4]:
df.head(3)

,wikipedia_id,freebase_id,title,author,publication_date,genres,synopsis
0,620,/m/0hhy,Animal Farm,George Orwell,1945-08-17,"{""/m/016lj8"": ""Roman \u00e0 clef"", ""/m/06nbt"":...","Old Major, the old boar on the Manor Farm, ca..."
1,843,/m/0k36,A Clockwork Orange,Anthony Burgess,1962,"{""/m/06n90"": ""Science Fiction"", ""/m/0l67h"": ""N...","Alex, a teenager living in near-future Englan..."
2,986,/m/0ldx,The Plague,Albert Camus,1947,"{""/m/02m4t"": ""Existentialism"", ""/m/02xlf"": ""Fi...",The text of The Plague is divided into five p...


## Missing values

`publication_date` is often missing but is **not used** by the recommender: it is dropped before removing incomplete rows, otherwise ~2,800 books would be lost for nothing.

In [5]:
df.isna().sum()

wikipedia_id           0
freebase_id            0
title                  0
author              2382
publication_date    5610
genres              3718
synopsis               0
dtype: int64

## Clean `genres` column

Genres are stored as a Freebase JSON dict `{freebase_id: name}`; we keep the sorted names.

In [6]:
df = df.drop(columns=["freebase_id", "publication_date"])
df["genres"] = df["genres"].apply(parse_genre)
df[["title", "genres"]].head()

,title,genres
0,Animal Farm,"Children's literature, Fiction, Roman à clef, ..."
1,A Clockwork Orange,"Fiction, Novella, Satire, Science Fiction, Spe..."
2,The Plague,"Absurdist fiction, Existentialism, Fiction, Novel"
3,An Enquiry Concerning Human Understanding,NaN
4,A Fire Upon the Deep,"Fantasy, Fiction, Hard science fiction, Scienc..."


In [7]:
# Genre distribution (one book can have several genres)
genre_counts = df["genres"].dropna().str.split(", ").explode().value_counts()
print(f"{len(genre_counts)} distinct genres")
genre_counts.head(20)

227 distinct genres


genres
Fiction                   4747
Speculative fiction       4314
Science Fiction           2870
Novel                     2463
Fantasy                   2413
Children's literature     2122
Mystery                   1396
Young adult literature     825
Suspense                   765
Crime Fiction              753
Historical novel           654
Thriller                   568
Horror                     511
Romance novel              435
Historical fiction         388
Detective fiction          341
Adventure novel            330
Non-fiction                230
Alternate history          226
Spy fiction                190
Name: count, dtype: int64

## Preprocess `synopsis`

Lower-case, tokenization, letters only, lemmatization, stop-word removal.

This normalized text is kept for **lexical** analyses (keywords, TF-IDF, Word2Vec...). SBERT, on the other hand, receives the **raw** synopsis (`synopsis_raw`): it is trained on natural sentences and needs stop-words, negations and word order to capture context.

In [8]:
download_require_nltk_resources()
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

df["synopsis_raw"] = df["synopsis"]
df["synopsis"] = df["synopsis"].apply(lambda x: preprocess_text(x, lemmatizer, stop_words))
df["synopsis"] = df["synopsis"].replace("", np.nan)  # empty after preprocessing = missing
df[["synopsis_raw", "synopsis"]].head(3)

,synopsis_raw,synopsis
0,"Old Major, the old boar on the Manor Farm, ca...",old major old boar manor farm call animal farm...
1,"Alex, a teenager living in near-future Englan...",alex teenager live england lead gang nightly o...
2,The text of The Plague is divided into five p...,text plague divide five part town oran thousan...


## Drop incomplete rows and duplicates

In [9]:
df = df.dropna()
# Same book listed twice (same title + author): keep the first entry
df = df.drop_duplicates(subset=["title", "author"]).reset_index(drop=True)
print(f"{len(df)} books kept")
df.info()

12040 books kept
<class 'pandas.DataFrame'>
RangeIndex: 12040 entries, 0 to 12039
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   wikipedia_id  12040 non-null  int64
 1   title         12040 non-null  str  
 2   author        12040 non-null  str  
 3   genres        12040 non-null  str  
 4   synopsis      12040 non-null  str  
 5   synopsis_raw  12040 non-null  str  
dtypes: int64(1), str(5)
memory usage: 50.6 MB


## Build the files used by the app

`run()` writes `data/processed/booksummaries.csv` and the embeddings (`data/embeddings/booksummaries.npz`: one vector for the synopsis, one for the genres, fused at load time). It does nothing if the files are already up to date; it re-encodes only if they are missing or out of sync (~1h30 on CPU with `all-mpnet-base-v2`, see notebook 02 for the choice).

In [10]:
run(root=ROOT)

[Data] Start data processing..
[Data] Assign macro-categories


[Data] Done


## Macro-categories (thematic blocks)

The 221 Freebase genres are grouped into **11 macro-categories** (`configs/categories.yaml`), plus a separate **Youth** audience. ~25% of the books only have vague genres ("Novel", "Fiction"): they receive the category whose **TF-IDF centroid** (mean TF-IDF vector of the books labeled by their genres) is the closest. In the benchmark of notebook 02, TF-IDF reached 0.700 accuracy for this task vs 0.655 for SBERT: deciding the theme of a book from its vocabulary is a lexical task. These categories are marked *inferred* in the app and never used as ground truth.

In [11]:
from src.data.data import load_books

books, vectors = load_books(root=ROOT)
print(books["category_source"].value_counts(), "\n")
print("Youth audience:", int(books["is_youth"].sum()), "books")
books["categories"].explode().value_counts()

category_source
genres       8998
predicted    3042
Name: count, dtype: int64 

Youth audience: 2489 books


categories
Science Fiction        3059
Fantasy                2664
Mystery & Crime        1874
Thriller & Suspense    1424
Literary Fiction       1420
Historical             1356
Romance                 945
Non-fiction             869
Horror & Gothic         663
Humour & Satire         656
Adventure               528
Name: count, dtype: int64